# Cloud object storage from the embedded engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.55.0/cookbook/notebooks/recipes/cloud_storage.ipynb)

Built from [`cookbook/recipes/cloud_storage/example.py`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/recipes/cloud_storage/example.py). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.55.0 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed samples with a compact text encoder: seconds
# to minutes, or for a chapter that fine-tunes an encoder, minutes on a GPU and up
# to hours on a CPU. The models it runs are downloaded from the Hugging Face Hub on
# first use. SCALE = "full" runs it over the published data and a larger text
# encoder instead, on the GPU: the chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.55.0", engine + "==0.55.0", "jammi-cookbook[cloud] @ git+https://github.com/f-inverse/jammi-ai@py-v0.55.0#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

Read a source straight out of an S3 bucket and write every result table the
session produces — its Parquet and its ANN index segments — back into the
bucket, from `jammi.connect("file://…")`. The embedded engine speaks the same
`s3://` / `gs://` / `azure://` / `r2://` URLs a `jammi-server` does.

**When to use this pattern.** Your data already lives in object storage, or
the results must outlive the machine that computed them — a notebook runtime,
a batch job on a spot instance — without standing up a server.

## The configuration

```toml
[storage]
result_root = "s3://jammi-cookbook/results"

[storage.cloud.s3]
region = "us-east-1"
endpoint = "http://127.0.0.1:<port>"  # a local S3-compatible server only
allow_http = true                     # a local S3-compatible server only
```

Pass it as `jammi.connect("file:///var/lib/jammi", config="jammi.toml")`. The
keys come from the environment (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`),
or from the SDK's credential chain against real S3. See
[Store Sources and Results in Cloud Object Storage](https://github.com/f-inverse/jammi-ai/blob/main/docs/guide/src/cloud-storage.md).

## API surface exercised

- `jammi.connect("file://…", config=…)` with a `[storage]` section
- `Session.add_source(name, url="s3://…", format="parquet")`
- `Session.generate_embeddings(...)`, `encode_query(...)`, `search(...)`

The embedded engine speaks the same storage URLs a server does. A source
registered at an `s3://` URL is read through the S3 driver, and a
`[storage] result_root` on `s3://` puts every result table the session
writes, its Parquet and its ANN index segments, in the bucket instead of on
local disk.

This program serves S3 locally from `moto`, an S3-compatible server, and
runs one search twice: over local disk, then with both the corpus and the
results in a bucket. The local run is the reference the bucket run must
match.

In [ ]:
import logging
import os
import socket
import tempfile
from pathlib import Path

import boto3
from moto.server import ThreadedMotoServer

import jammi
from jammi_cookbook import fixtures

os.environ.setdefault("JAMMI_ENGINE__BATCH_SIZE", "8")

CORPUS = fixtures.path("tiny_corpus.parquet")
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BUCKET = "jammi-cookbook"
QUERY = "how does quantum computing work?"

## Serve S3 locally and fill a bucket

The S3 driver reads its keys from the environment, as it does against real
S3. Against real S3, the SDK's credential chain (environment variables, an
instance profile, …) supplies them.

In [ ]:
with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
endpoint = f"http://127.0.0.1:{port}"

os.environ.update(
    AWS_ACCESS_KEY_ID="cookbook", AWS_SECRET_ACCESS_KEY="cookbook", AWS_REGION="us-east-1"
)
# The local S3 server logs every request; this program's own output is the story.
logging.getLogger("werkzeug").setLevel(logging.ERROR)
server = ThreadedMotoServer(ip_address="127.0.0.1", port=port)
server.start()

s3 = boto3.client("s3", endpoint_url=endpoint, region_name="us-east-1")
s3.create_bucket(Bucket=BUCKET)
s3.upload_file(str(CORPUS), BUCKET, "sources/tiny_corpus.parquet")

## One program, wherever its data lives

Register the corpus at a URL, embed it, search it, and return the ranked
`(_row_id, similarity)` pairs.

In [ ]:
def search_rows(target_dir: str, *, corpus_url: str, config: Path | None) -> list[tuple]:
    with jammi.connect(f"file://{target_dir}", config=str(config) if config else None) as db:
        db.add_source("corpus", url=corpus_url, format="parquet")
        db.generate_embeddings(
            source="corpus", model=MODEL, columns=["content"], key="id", modality="text"
        )
        query = db.encode_query(model=MODEL, query=QUERY)
        hits = db.search("corpus", query=query, k=5).to_pylist()
        return [(row["_row_id"], round(row["similarity"], 6)) for row in hits]


golden = search_rows(tempfile.mkdtemp(), corpus_url=str(CORPUS), config=None)
print(f"top-5 over local disk: {golden}")

## Run it against the bucket

Where results go, and how to reach the store, is deployment configuration:
the same `[storage]` section a server reads. Against real S3, drop the
`endpoint` and `allow_http` lines.

In [ ]:
cloud = Path(tempfile.mkdtemp())
config = cloud / "jammi.toml"
config.write_text(
    f'[storage]\nresult_root = "s3://{BUCKET}/results"\n\n'
    f'[storage.cloud.s3]\nregion = "us-east-1"\nendpoint = "{endpoint}"\n'
    "allow_http = true\n"
)
from_s3 = search_rows(
    str(cloud), corpus_url=f"s3://{BUCKET}/sources/tiny_corpus.parquet", config=config
)
print(f"top-5 over s3://{BUCKET}: {from_s3}")
assert from_s3 == golden, f"the S3 run ranked differently: {from_s3} vs {golden}"

## The results are in the bucket

The result table's Parquet and its index segments were written under the
bucket's `results/` prefix.

In [ ]:
objects = [o["Key"] for o in s3.list_objects_v2(Bucket=BUCKET)["Contents"]]
server.stop()

results = sorted(key for key in objects if key.startswith("results/"))
for key in results:
    print(f"s3://{BUCKET}/{key}")
assert any(key.endswith(".parquet") for key in results), results
assert any(key.endswith(".usearch") for key in results), results